In [1]:
import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

In [18]:
import torch
import evaluate
import numpy as np
from peft import UIOrthoLoRAConfig, UILinLoRAConfig, get_peft_model, TaskType, PeftConfig, PeftModel
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import load_dataset

In [29]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [1]:
# Load GPT-2 tokenizer with use_fast=False for full control
tok = AutoTokenizer.from_pretrained("gpt2", use_fast=False)

# BEFORE: GPT-2 has no pad token
print("Before:", tok.pad_token, tok.pad_token_id)

# Add a real PAD token (ID will be 50257)
tok.add_special_tokens({"pad_token": "[PAD]"})

print("After:", tok.pad_token, tok.pad_token_id)
print("PAD token ID:", tok.convert_tokens_to_ids("[PAD]"))

# Create a short batch of prompt_ids
examples = [
    tok.convert_tokens_to_ids(tok.tokenize("name[Bibimbap House] =>")),
    tok.convert_tokens_to_ids(tok.tokenize("name[Wildwood] => food[Italian]"))
]

# Left pad manually
pad_id = tok.pad_token_id
max_len = max(len(x) for x in examples)

for i, seq in enumerate(examples):
    pad = [pad_id] * (max_len - len(seq))
    padded = pad + seq
    print(f"\nExample {i+1}")
    print("Raw IDs:     ", padded)
    print("Tokens:      ", tok.convert_ids_to_tokens(padded))
    print("Last token:  ", padded[-1], tok.convert_ids_to_tokens([padded[-1]])[0])
    print("Padding OK?  ", padded[-1] != pad_id)


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Before: None None
After: [PAD] 50257
PAD token ID: 50257

Example 1
Raw IDs:      [50257, 3672, 58, 33, 571, 14107, 499, 2097, 60, 5218]
Tokens:       ['[PAD]', 'name', '[', 'B', 'ib', 'imb', 'ap', 'ĠHouse', ']', 'Ġ=>']
Last token:   5218 Ġ=>
Padding OK?   True

Example 2
Raw IDs:      [3672, 58, 25946, 3822, 60, 5218, 2057, 58, 45696, 60]
Tokens:       ['name', '[', 'Wild', 'wood', ']', 'Ġ=>', 'Ġfood', '[', 'Italian', ']']
Last token:   60 ]
Padding OK?   True


/opt/anaconda3/envs/guyb_env2/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [5]:
from datasets import load_dataset

ds = load_dataset("tuetschek/e2e_nlg")

# View a few examples
for i in range(10):
    record = ds["test"][i]
    print("Meaning Representation (MR):", record["meaning_representation"])
    print("Reference:", record.get("human_reference") or record.get("reference"))
    print("-" * 60)


Meaning Representation (MR): name[Blue Spice], eatType[coffee shop], area[city centre]
Reference: A coffee shop in the city centre area called Blue Spice.
------------------------------------------------------------
Meaning Representation (MR): name[Blue Spice], eatType[coffee shop], area[city centre]
Reference: Blue Spice is a coffee shop in city centre.
------------------------------------------------------------
Meaning Representation (MR): name[Blue Spice], eatType[coffee shop], area[riverside]
Reference: There is a coffee shop Blue Spice in the riverside area.
------------------------------------------------------------
Meaning Representation (MR): name[Blue Spice], eatType[coffee shop], area[riverside]
Reference: At the riverside, there is a coffee shop called The Blue Spice.
------------------------------------------------------------
Meaning Representation (MR): name[Blue Spice], eatType[coffee shop], customer rating[5 out of 5], near[Crowne Plaza Hotel]
Reference: The coffee s

In [4]:
len(ds['test'])

4693

In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load test set
ds = load_dataset("tuetschek/e2e_nlg")
test_ds = ds["test"]

# Load tokenizer (use the same as your model)
tokenizer = AutoTokenizer.from_pretrained("gpt2")  # or your model's tokenizer

# Detokenize model predictions
def decode_preds(preds):
    return tokenizer.batch_decode(preds, skip_special_tokens=True)

# Print model generations vs gold labels
def print_preds_vs_gold(model_preds, labels):
    for i in range(min(10, len(model_preds))):
        pred = model_preds[i].strip()
        gold = labels[i].strip()
        print(f"💡 Prediction {i+1}: {pred}")
        print(f"✅ Gold Label {i+1}: {gold}")
        print("-" * 80)


/opt/anaconda3/envs/guyb_env2/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['meaning_representation', 'human_reference'],
        num_rows: 42061
    })
    validation: Dataset({
        features: ['meaning_representation', 'human_reference'],
        num_rows: 4672
    })
    test: Dataset({
        features: ['meaning_representation', 'human_reference'],
        num_rows: 4693
    })
})

In [7]:
for i, sentence in enumerate(ds["test"]["human_reference"]):
    print(f"Sentence {i+1}: {sentence}")
    print("-" * 80)

    if i == 20:
        break

Sentence 1: A coffee shop in the city centre area called Blue Spice.
--------------------------------------------------------------------------------
Sentence 2: Blue Spice is a coffee shop in city centre.
--------------------------------------------------------------------------------
Sentence 3: There is a coffee shop Blue Spice in the riverside area.
--------------------------------------------------------------------------------
Sentence 4: At the riverside, there is a coffee shop called The Blue Spice.
--------------------------------------------------------------------------------
Sentence 5: The coffee shop Blue Spice is based near Crowne Plaza Hotel and has a high customer rating of 5 out of 5.
--------------------------------------------------------------------------------
Sentence 6: The Blue Spice coffee shop, near Crowne Plaza Hotel, has a customer rating of 5 out of 5.
--------------------------------------------------------------------------------
Sentence 7: If you want 

In [19]:
from torch.utils.data import DataLoader
INFERENCE_ARGS = {
    "num_beams": 10,
    "no_repeat_ngram_size": 4,
    "length_penalty": 0.9,
    "max_new_tokens": 64,
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [20]:
def set_tokenizer(tokenizer):
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

def set_contiguous(model):
    for m in model.modules():
        if hasattr(m, "parametrizations") and "weight" in m.parametrizations:
            base = m.parametrizations.weight[0].base
            if not base.is_contiguous():
                base.data = base.data.contiguous()

def get_tokenizer_and_model(model_path: str, device):
    """
    Load a base model and inject a saved PEFT adapter from `model_path`.
    """
    # 1) Load the adapter config to get the original base model
    peft_config = PeftConfig.from_pretrained(model_path)
    base_model_name = peft_config.base_model_name_or_path

    # 2) Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=False)
    set_tokenizer(tokenizer)

    base_model = AutoModelForCausalLM.from_pretrained(base_model_name)
    base_model.config.pad_token_id = tokenizer.pad_token_id
    base_model.generation_config.pad_token_id = tokenizer.pad_token_id
    base_model = base_model.to(device)

    # 3) Load the adapter into the base model
    model = PeftModel.from_pretrained(base_model, model_path)
    model = model.to(device)

    # 4) Ensure contiguous weights (optional)
    set_contiguous(model)

    return tokenizer, model, peft_config

In [21]:
tokenizer, model, peft_config = get_tokenizer_and_model("outputs/models/lr_5e-3", device)

ValueError: Can't find 'adapter_config.json' at 'outputs/models/lr_5e-3'

In [10]:
def collate_fn(batch):
        feats = [{"input_ids": b["prompt_ids"]} for b in batch]
        out = tokenizer.pad(
        feats,
        padding="longest",
        return_attention_mask=True,
        return_tensors="pt"
    )
        return out        

dataloader = DataLoader(
    ds["test"],
    batch_size=8,
    collate_fn=collate_fn,
)

for item in tqdm(dataloader):
    prompt_ids = item["input_ids"].to(device)
    attention_mask = item["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=prompt_ids,
            attention_mask=attention_mask,
            max_new_tokens=INFERENCE_ARGS["max_new_tokens"],
            num_beams=INFERENCE_ARGS["num_beams"],
            no_repeat_ngram_size=INFERENCE_ARGS["no_repeat_ngram_size"],
            length_penalty=INFERENCE_ARGS["length_penalty"],
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    # strip the prompt text (“=>” part) to isolate hypothesis
    preds = [p.split("=>")[-1].strip() for p in preds]
    gen_texts.extend(preds)

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 11)

In [3]:
def load_and_prepare(tokenizer, max_length=128):
    """Load E2E dataset and prepare tokenised fields."""
    ds = load_dataset("tuetschek/e2e_nlg")

    def linearise(record):
        mr = record["meaning_representation"]  # e.g. "name[Bibimbap House], food[Indian]"
        ref = record["human_reference"] if "human_reference" in record else record["reference"]
        prompt = f"{mr} => "  # simple prompt pattern
        example = prompt + ref
        tokenised = tokenizer(
            example,
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        labels = tokenised["input_ids"].copy()
        # Mask prompt tokens so they are ignored in the loss (label = -100)
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        prompt_len = len(prompt_ids)
        labels[:prompt_len] = [-100] * prompt_len
        tokenised["labels"] = labels
        return tokenised

    ds = ds.map(linearise, remove_columns=ds["train"].column_names)
    return ds

In [4]:
from pycocoevalcap.cider.cider import Cider

class CiderMetric:
    """Wraps pycocoevalcap so it looks like an `evaluate` metric."""
    def __init__(self):
        self.scorer = Cider()

    def compute(self, *, predictions, references):
        # pycocoevalcap expects dicts: {idx: ["sentence"]}
        hyps = {i: [pred] for i, pred in enumerate(predictions)}
        refs = {i: [ref]  for i, ref  in enumerate(references)}
        score, _ = self.scorer.compute_score(refs, hyps)
        return {"cider": score}

cider_metric = CiderMetric()

In [5]:
bleu_metric = evaluate.load("sacrebleu")
meteor_metric = evaluate.load("meteor")
rouge_metric = evaluate.load("rouge")
nist_metric = evaluate.load("nist_mt")

[nltk_data] Downloading package wordnet to
[nltk_data]     /home/guy.bilitski/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /home/guy.bilitski/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /home/guy.bilitski/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [6]:
def postprocess_text(preds, labels):
    preds = [p.strip() for p in preds]
    labels = [l.strip() for l in labels]
    return preds, labels


def set_contiguous(model):
    for m in model.modules():
        if hasattr(m, "parametrizations") and "weight" in m.parametrizations:
            base = m.parametrizations.weight[0].base
            if not base.is_contiguous():
                base.data = base.data.contiguous()


# ------------------------------------------------------------------
# helper -----------------------------------------------------------
def _postprocess_strs(predictions, references):
    """Strip leading/trailing spaces & unify whitespace."""
    preds = [p.strip() for p in predictions]
    refs  = [r.strip() for r in references]
    return preds, refs
# ------------------------------------------------------------------



def compute_metrics(eval_pred):
    """Compute BLEU, METEOR, ROUGE-L, (optionally) NIST on E2E-NLG."""
    preds, labels = eval_pred

    # ── tensors → numpy ───────────────────────────────────────────────
    if isinstance(preds, torch.Tensor):
        preds = preds.cpu().numpy()
    if isinstance(labels, torch.Tensor):
        labels = labels.cpu().numpy()

    # ── logits → ids if necessary ─────────────────────────────────────
    if preds.ndim == 3:                      # (batch, seq, vocab)
        preds = preds.argmax(-1)

    # ── un-mask labels ────────────────────────────────────────────────
    labels = labels.copy()
    labels[labels == -100] = tokenizer.pad_token_id

    # ── decode ────────────────────────────────────────────────────────
    pred_strs  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    label_strs = tokenizer.batch_decode(labels, skip_special_tokens=True)
    pred_strs  = [s.strip() for s in pred_strs]
    label_strs = [s.strip() for s in label_strs]

    # ── metrics ───────────────────────────────────────────────────────
    bleu   = bleu_metric.compute(
                predictions=pred_strs,
                references=[[r] for r in label_strs]
             )["score"]

    meteor = meteor_metric.compute(
                predictions=pred_strs,
                references=label_strs
             )["meteor"]

    rougeL = rouge_metric.compute(
                predictions=pred_strs,
                references=label_strs,
                use_stemmer=True
             )["rougeL"]

    cider = cider_metric.compute(
                predictions=pred_strs,
                references=label_strs   # wrapper handles dict-conversion
            )["cider"]

    # ---- NIST (may not exist on tiny samples) ------------------------
    nist_raw = nist_metric.compute(
                predictions=pred_strs,
                references=[[r] for r in label_strs]
              )
    nist_val = nist_raw.get("nist", nist_raw.get("score"))  # could be None

    # helper to round only numerics
    def _r(x):
        return round(float(x), 4) if isinstance(x, Number) else x

    out = {
        "bleu"  : _r(bleu),
        "meteor": _r(meteor),
        "rougeL": _r(rougeL),
        "cider"  : _r(cider),
    }
    if nist_val is not None:
        out["nist"] = _r(nist_val)

    return out


In [7]:
orthoLoRAConfig = UIOrthoLoRAConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    fan_in_fan_out         = True,   # GPT-2 matrices are (out, in)
    initial_scaler         = 0.1,    # scale of the diagonal Σ at init
    initial_sigma          = 0.1,    # std-dev for the trainable Σ entries
    uiortholora_alpha      = 1,
    uiortholora_dropout    = 0,
    num_svalues_to_adapt   = 2,       # adapt the top-4 singular values
    num_svectors_to_adapt  = 2,       # adapt the corresponding vectors
    task_type              = TaskType.CAUSAL_LM
)

In [8]:
from peft import LoraConfig

lora_config = LoraConfig(
    target_modules=["attn.c_attn", "attn.c_proj"],
    r=2,                         # very low rank for easy debugging
    lora_alpha=1,               # no extra scaling
    lora_dropout=0.0,           # no dropout for deterministic behavior
    bias="none",                # keep bias untouched
    fan_in_fan_out=False,       # match GPT-2 shape: (out, in)
    task_type=TaskType.CAUSAL_LM
)


In [16]:
model_path = "gpt2-medium"
seed=42

torch.manual_seed(seed)
np.random.seed(seed)


tokenizer = AutoTokenizer.from_pretrained(model_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

ds = load_and_prepare(tokenizer)

base_model = AutoModelForCausalLM.from_pretrained(model_path)
base_model.config.pad_token_id = tokenizer.pad_token_id

Using the latest cached version of the module from /home/guy.bilitski/.cache/huggingface/modules/datasets_modules/datasets/tuetschek--e2e_nlg/bfeceb720929c2705bd227d1cfe5eaaab102a0bdac10dad618dac1e00c737430 (last modified on Sat Jun 21 16:13:44 2025) since it couldn't be found locally at tuetschek/e2e_nlg, or remotely on the Hugging Face Hub.


In [26]:
peft_config = PeftConfig.from_pretrained("outputs/models")

In [30]:
base_model_check = AutoModelForCausalLM.from_pretrained(peft_config.base_model_name_or_path)
base_model_check = base_model_check.to(device)

In [31]:
model = PeftModel.from_pretrained(base_model_check, "outputs/models")

In [ ]:
stop

In [18]:
base_model = base_model.to(device)


In [19]:
base_model.device

device(type='cuda', index=0)

In [12]:
stop

NameError: name 'stop' is not defined

In [20]:
model = get_peft_model(base_model, orthoLoRAConfig)

In [21]:
set_contiguous(model)
model.print_trainable_parameters()

trainable params: 147,936 || all params: 354,971,104 || trainable%: 0.0417


In [22]:
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="outputs/check",
    overwrite_output_dir=True,
    eval_strategy="no",
    save_strategy="no",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    eval_accumulation_steps=2,
    learning_rate=1e-3,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=50,
    save_total_limit=1,
    report_to="none",
)

In [23]:
trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=ds["train"].select(range(500)),
        eval_dataset=ds["validation"],
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [24]:
# trainer.train()
# from accelerate import Accelerator
# accelerator = Accelerator()
# trainer = accelerator.prepare(trainer)
trainer.train()


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,4.350600
100,4.257500
150,4.217100
200,4.127900
250,4.074500
300,4.017300


TrainOutput(global_step=315, training_loss=4.168453349764385, metrics={'train_runtime': 316.8974, 'train_samples_per_second': 7.889, 'train_steps_per_second': 0.994, 'total_flos': 580721971200000.0, 'train_loss': 4.168453349764385, 'epoch': 5.0})

In [25]:
trainer.save_model("outputs/models")

In [ ]:
stop

In [16]:
# Load the trained model from saved checkpoint
model = AutoModelForCausalLM.from_pretrained("outputs/models")
model = model.to("cuda")

In [18]:
# from pathlib import Path
# import json

# metrics = trainer.evaluate(ds["test"].select(range(100)))
# Path(training_args.output_dir).mkdir(parents=True, exist_ok=True)
# (Path(training_args.output_dir) / "test_metrics.json").write_text(json.dumps(metrics, indent=2))
# print("Test metrics saved to", training_args.output_dir)


KeyboardInterrupt: 

In [24]:
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")


eval_loss: 4.2372
eval_model_preparation_time: 0.0220
eval_bleu: 13.5479
eval_meteor: 0.3804
eval_rougeL: 0.3445
eval_cider: 0.2929
eval_runtime: 420.6666
eval_samples_per_second: 2.3770
eval_steps_per_second: 0.2970


In [25]:
# first_block = model.base_model.model.transformer.h[0]         # First transformer block
# attn = first_block.attn                                       # Attention module
# c_attn = attn.c_attn                                          # The fused QKV projection
# print(f"c_attn.weight shape: {c_attn.weight.shape}")          # Should be [3072, 1024]

In [2]:
import torch
import time


In [5]:
a = torch.randn(1024)
# b = torch.randn(1024)
b = torch.ones(1024)
rand_matrix = torch.randn(1024, 1024)

In [15]:
a = torch.tensor([1,2,3,4,5])
b = torch.tensor([10,11,12])

In [23]:
c = torch.tensor([[1,2,3],[4,5,6]])

In [29]:
c

tensor([[1, 2, 3],
        [4, 5, 6]])

In [28]:
c @ torch.diag(torch.tensor([1,2,3]))

tensor([[ 1,  4,  9],
        [ 4, 10, 18]])

In [20]:
ab = torch.outer(a,b)

In [21]:
print(ab)

tensor([[10, 11, 12],
        [20, 22, 24],
        [30, 33, 36],
        [40, 44, 48],
        [50, 55, 60]])


In [28]:
torch.allclose(torch.mul(rand_matrix, ab), (a[:, None] * rand_matrix) * b[None, :], atol=1e-5, rtol=1e-5)

True

In [39]:
torch.allclose(rand_matrix * b.unsqueeze(1) * a.unsqueeze(0), b.unsqueeze(1) * rand_matrix * a.unsqueeze(0), atol=1e-5, rtol=1e-5)

True

In [46]:
torch.allclose(rand_matrix + rand_matrix * a.unsqueeze(1) * b.unsqueeze(0),
                torch.mul(1+ab, rand_matrix), atol=1e-5, rtol=1e-5)

True

In [40]:
def naive_calc(a, b, rand_matrix):
    return b.unsqueeze(1) * rand_matrix * a.unsqueeze(0)

In [29]:
def vectorized(a, b, rand_matrix):
    ab = torch.outer(a,b)
    return torch.mul(rand_matrix, ab)

In [49]:
def calc_major1(a, b, rand_matrix):
    return rand_matrix + rand_matrix * a.unsqueeze(1) * b.unsqueeze(0)

In [50]:
def calc_major2(a, b, rand_matrix):
    ab = torch.outer(a,b)
    return torch.mul(1+ab, rand_matrix)

In [51]:
times = []
for _ in range(1000):
    torch.cuda.synchronize()
    start = time.time()
    calc_major1(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"calc_major1 Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

times = []
for _ in range(1000):
    torch.cuda.synchronize()
    start = time.time()
    calc_major2(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"calc_major2 Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

calc_major1 Avg: 0.000602 sec | Std: 0.003167
calc_major2 Avg: 0.002776 sec | Std: 0.002650


In [41]:
times = []
for _ in range(10000):
    torch.cuda.synchronize()
    start = time.time()
    naive_calc(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"naive_calc Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

naive_calc Avg: 0.007375 sec | Std: 0.006838


In [34]:
times = []
for _ in range(1000):
    torch.cuda.synchronize()
    start = time.time()
    vectorized(a, b, rand_matrix)
    torch.cuda.synchronize()
    end = time.time()
    times.append(end - start)

print(f"vectorized Avg: {sum(times)/len(times):.6f} sec | Std: {torch.std(torch.tensor(times)).item():.6f}")

vectorized Avg: 0.011807 sec | Std: 0.006341


In [ ]:
torch.norm